# 01 — PaddlePaddle Baseline

Runs PP-LCNet_x1_0_doc_ori using the native PaddlePaddle inference engine.
This is the **ground-truth baseline** — accuracy and latency all other runtimes are compared against.

Model download: https://paddlepaddle.github.io/PaddleOCR/latest/en/ppstructure/models_list.html

Expected files in `models/paddle/`:
- `inference.pdmodel`
- `inference.pdiparams`

In [ ]:
import sys
sys.path.insert(0, '..')

import time
from pathlib import Path

import numpy as np
import pandas as pd
import paddle
from paddle.inference import Config, create_predictor
from tqdm import tqdm

from src.preprocess import load_and_preprocess

MODEL_DIR = Path('../models/paddle')
DATASET_CSV = Path('../data/dataset.csv')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

print(f'PaddlePaddle version: {paddle.__version__}')

In [ ]:
# --- Download model if not present ---
# Un-comment and run once
# import urllib.request, zipfile
# url = 'https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/doc_orientation_classify_v3_infer.tar'
# urllib.request.urlretrieve(url, '../models/doc_ori.tar')
# import tarfile
# with tarfile.open('../models/doc_ori.tar') as t:
#     t.extractall('../models/paddle')

In [ ]:
def build_predictor(model_dir: Path):
    config = Config(
        str(model_dir / 'inference.pdmodel'),
        str(model_dir / 'inference.pdiparams'),
    )
    config.disable_gpu()
    config.enable_mkldnn()
    config.set_cpu_math_library_num_threads(4)
    config.switch_ir_optim(True)
    return create_predictor(config)

predictor = build_predictor(MODEL_DIR)
input_names = predictor.get_input_names()
output_names = predictor.get_output_names()
print('Inputs:', input_names)
print('Outputs:', output_names)

In [ ]:
def predict(image_path: str) -> tuple[int, float]:
    tensor = load_and_preprocess(image_path)
    input_handle = predictor.get_input_handle(input_names[0])
    input_handle.reshape(tensor.shape)
    input_handle.copy_from_cpu(tensor)
    predictor.run()
    output = predictor.get_output_handle(output_names[0]).copy_to_cpu()
    pred_label = int(np.argmax(output, axis=1)[0])
    confidence = float(np.max(output))
    return pred_label, confidence

In [ ]:
df = pd.read_csv(DATASET_CSV)

predictions, confidences, latencies = [], [], []

for row in tqdm(df.itertuples(), total=len(df), desc='PaddlePaddle inference'):
    t0 = time.perf_counter()
    pred, conf = predict(row.image_path)
    latencies.append((time.perf_counter() - t0) * 1000)  # ms
    predictions.append(pred)
    confidences.append(conf)

df['paddle_pred'] = predictions
df['paddle_conf'] = confidences
df['paddle_latency_ms'] = latencies
df.to_csv(RESULTS_DIR / 'paddle_results.csv', index=False)
print('Saved results/paddle_results.csv')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

accuracy = (df['paddle_pred'] == df['label']).mean()
avg_latency = df['paddle_latency_ms'].mean()
throughput = 1000 / avg_latency

print(f'Accuracy:         {accuracy:.4f}')
print(f'Avg latency:      {avg_latency:.2f} ms')
print(f'Throughput:       {throughput:.1f} img/s')
print()
print(classification_report(df['label'], df['paddle_pred'],
                             target_names=['0deg','90deg','180deg','270deg']))

cm = confusion_matrix(df['label'], df['paddle_pred'])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', ax=ax,
            xticklabels=['0','90','180','270'],
            yticklabels=['0','90','180','270'])
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('PaddlePaddle — Confusion Matrix')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'paddle_confusion_matrix.png', dpi=150)
plt.show()